# Распознавание дорожных знаков свёрточной нейронной сетью (GTSRB)

Эксплуатационная практика. Предметная область — оценка дорожной ситуации / классификация изображений.

**Задача:** по изображению дорожного знака определить, к какому из 43 классов он относится.
Такая задача решается в системах помощи водителю (ADAS), беспилотных автомобилях и мобильных роботах.

**Данные:** German Traffic Sign Recognition Benchmark (GTSRB) — открытый набор, собранный
Институтом нейроинформатики Рурского университета (Бохум) по видеозаписям с камеры автомобиля
на дорогах Германии; использовался в соревновании IJCNN 2011.

План блокнота:
1. Подготовка среды
2. Загрузка данных и разведочный анализ
3. Предобработка: разбиение, приведение размера, стандартизация, аугментация
4. Архитектура сети и расчёт слоёв
5. Базовая модель
6. Эксперименты (функции активации, BatchNorm, Dropout, аугментация, скорость обучения, глубина)
7. Итоговая модель и оценка на тестовой выборке

## 1. Подготовка среды

In [ ]:
# В Google Colab: Среда выполнения -> Сменить среду выполнения -> GPU (T4).
# Укажите адрес своего репозитория на GitHub:
REPO_URL = "https://github.com/ВАШ_ЛОГИН/gtsrb-traffic-signs.git"

import os, sys
from pathlib import Path

if "google.colab" in sys.modules and not Path("src").exists():
    repo_dir = Path("/content/gtsrb-traffic-signs")
    if not repo_dir.exists():
        !git clone {REPO_URL} {repo_dir}
    os.chdir(repo_dir)
elif Path.cwd().name == "notebooks":
    os.chdir("..")   # при локальном запуске работаем из корня репозитория
sys.path.insert(0, str(Path.cwd()))
print("Рабочая папка:", Path.cwd())

In [ ]:
import json
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import classification_report, f1_score, precision_recall_fscore_support

from src import config
from src.data import (load_gtsrb_file_lists, split_train_validation_by_track, track_id_from_path,
                      load_or_cache_arrays, compute_channel_mean_std, build_augmentation,
                      TrafficSignDataset, make_loader)
from src.models import TrafficSignCNN, count_parameters, layer_summary
from src.train import set_seed, get_device, run_experiment, evaluate
from src import plots

# Быстрая проверка кода на маленькой подвыборке (для отладки). Для настоящих результатов — False.
QUICK_TEST = os.environ.get("GTSRB_QUICK") == "1"
EXPERIMENT_EPOCHS = 2 if QUICK_TEST else 15
FINAL_EPOCHS = 2 if QUICK_TEST else 30

set_seed(config.SEED)
device = get_device()
print("PyTorch", torch.__version__, "| устройство:", device,
      "|", torch.cuda.get_device_name(0) if device.type == "cuda" else "")
pd.set_option("display.max_rows", 100)

## 2. Загрузка данных и разведочный анализ

Набор скачивается с официального зеркала (`torchvision.datasets.GTSRB`), ~350 МБ.
Все изображения один раз читаются в память и приводятся к размеру 32×32; результат
кэшируется в `data/`, чтобы при повторном запуске не читать 50 тыс. файлов заново.

In [ ]:
for folder in [config.DATA_DIR, config.RESULTS_DIR, config.FIGURES_DIR, config.EXPERIMENTS_DIR]:
    folder.mkdir(parents=True, exist_ok=True)
train_files, test_files = load_gtsrb_file_lists(config.DATA_DIR, download=True)
if QUICK_TEST:
    rng = np.random.default_rng(0)
    train_files = [train_files[i] for i in sorted(rng.choice(len(train_files), min(3000, len(train_files)), replace=False))]
    test_files = [test_files[i] for i in sorted(rng.choice(len(test_files), min(1000, len(test_files)), replace=False))]
print(f"Обучающая часть: {len(train_files)} изображений")
print(f"Тестовая часть:  {len(test_files)} изображений")
print("Пример файла:", train_files[0])

In [ ]:
suffix = "_quick" if QUICK_TEST else ""
train_arrays = load_or_cache_arrays(train_files, config.DATA_DIR / f"train_{config.IMAGE_SIZE}{suffix}.pt")
test_arrays = load_or_cache_arrays(test_files, config.DATA_DIR / f"test_{config.IMAGE_SIZE}{suffix}.pt")
print("Тензор изображений:", tuple(train_arrays.images.shape), train_arrays.images.dtype)

### 2.1. Распределение классов

In [ ]:
train_counts = np.bincount(train_arrays.labels.numpy(), minlength=config.NUM_CLASSES)
test_counts = np.bincount(test_arrays.labels.numpy(), minlength=config.NUM_CLASSES)
plots.plot_class_distribution(train_counts, test_counts)

class_table = pd.DataFrame({"Класс": config.CLASS_NAMES_RU, "Обучение": train_counts, "Тест": test_counts})
print(f"Самый частый класс: {class_table['Обучение'].max()} изобр., самый редкий: {class_table['Обучение'].min()} изобр. "
      f"(разница в {class_table['Обучение'].max() / max(1, class_table['Обучение'].min()):.1f} раза)")
class_table.sort_values("Обучение")

### 2.2. Примеры изображений

In [ ]:
plots.plot_class_examples(train_arrays.images, train_arrays.labels.numpy())

### 2.3. Размеры и яркость исходных изображений

In [ ]:
sizes = train_arrays.original_sizes
print(f"Ширина: от {sizes[:,0].min()} до {sizes[:,0].max()} пикс., медиана {np.median(sizes[:,0]):.0f}")
print(f"Высота: от {sizes[:,1].min()} до {sizes[:,1].max()} пикс., медиана {np.median(sizes[:,1]):.0f}")
print(f"Доля изображений с яркостью < 50 (очень тёмные): {(train_arrays.mean_brightness < 50).mean():.1%}")
plots.plot_size_and_brightness(sizes, train_arrays.mean_brightness)

### 2.4. Структура «треков»

Обучающая часть состоит из треков: каждый трек — около 30 последовательных кадров одного и того же
физического знака, снятого при приближении автомобиля. Это важно для разбиения на обучающую
и валидационную выборки (см. раздел 3).

In [ ]:
track_ids = np.array([f"{label}_{track_id_from_path(path)}" for path, label in train_files])
unique_tracks, track_sizes = np.unique(track_ids, return_counts=True)
print(f"Треков: {len(unique_tracks)}, кадров в треке: медиана {np.median(track_sizes):.0f}")
example_track = unique_tracks[0]
plots.plot_track_example(train_arrays.images, np.flatnonzero(track_ids == example_track),
                         f"Кадры одного трека (класс {example_track.split('_')[0]}): один знак с разного расстояния")

## 3. Предобработка

1. **Разбиение.** Официальная тестовая часть (12 630 изобр.) используется только для итоговой оценки.
   Обучающая часть делится на обучающую (≈85%) и валидационную (≈15%) выборки **по трекам**:
   если разбить случайно по кадрам, почти одинаковые кадры одного знака попадут в обе выборки,
   и точность на валидации будет завышена.
2. **Размер 32×32.** Исходные размеры от ~25 до ~250 пикселей; сети нужен вход фиксированного размера.
   32×32 достаточно, чтобы различить символы на знаке, и делает обучение быстрым.
3. **Стандартизация каналов:** x' = (x − μ_c) / σ_c, где μ_c и σ_c считаются только по обучающей выборке.
   Признаки в одном масштабе с центром в нуле ускоряют сходимость градиентного спуска.
4. **Аугментация** (только для обучения): поворот ±10°, сдвиг до 10%, масштаб 0.9–1.1,
   изменение яркости/контраста. Без горизонтального отражения — оно меняет смысл знаков «налево/направо».

In [ ]:
train_idx, val_idx = split_train_validation_by_track(train_files, config.VALIDATION_FRACTION, config.SEED)
val_tracks, train_tracks = set(track_ids[val_idx]), set(track_ids[train_idx])
print(f"Обучающая выборка: {len(train_idx)} изобр., валидационная: {len(val_idx)} изобр., тестовая: {len(test_files)}")
print("Общих треков у train и validation:", len(train_tracks & val_tracks))

train_images, train_labels = train_arrays.images[train_idx], train_arrays.labels[train_idx]
val_images, val_labels = train_arrays.images[val_idx], train_arrays.labels[val_idx]

mean, std = compute_channel_mean_std(train_images)
print("Среднее по каналам RGB:", np.round(mean, 4), "| ст. отклонение:", np.round(std, 4))
with open(config.RESULTS_DIR / "normalization.json", "w") as f:
    json.dump({"mean": mean, "std": std}, f)

In [ ]:
sample = train_images[:2000]
normalized = TrafficSignDataset(sample, train_labels[:2000], mean, std)
normalized_batch = torch.stack([normalized[i][0] for i in range(len(normalized))])
plots.plot_before_after_normalization(sample, normalized_batch)

augmentation = build_augmentation()
torch.manual_seed(config.SEED)
plots.plot_augmentation_examples(train_images, augmentation, indices=np.linspace(0, len(train_images) - 1, 4, dtype=int).tolist())

In [ ]:
train_dataset_plain = TrafficSignDataset(train_images, train_labels, mean, std)
train_dataset_augmented = TrafficSignDataset(train_images, train_labels, mean, std, augmentation=augmentation)
val_dataset = TrafficSignDataset(val_images, val_labels, mean, std)
test_dataset = TrafficSignDataset(test_arrays.images, test_arrays.labels, mean, std)

train_loader_plain = make_loader(train_dataset_plain, shuffle=True)
train_loader_augmented = make_loader(train_dataset_augmented, shuffle=True)
val_loader = make_loader(val_dataset, shuffle=False)
test_loader = make_loader(test_dataset, shuffle=False)

## 4. Архитектура сети и расчёт слоёв

Выбрана **свёрточная нейронная сеть (CNN)**. Свёрточный слой применяет один и тот же небольшой фильтр
ко всем участкам изображения, поэтому:
* находит локальные признаки (края, углы, цветные контуры, цифры) независимо от их положения;
* имеет на порядки меньше параметров, чем полносвязный слой на тех же входах;
* слои образуют иерархию: первые выделяют простые признаки, следующие — их комбинации (форма знака, символ).

Базовая архитектура — три блока `Conv 3×3 → ReLU → MaxPool 2×2` (32 → 16 → 8 → 4 пикселя;
3 → 32 → 64 → 128 каналов) и классификатор `Linear(2048→256) → ReLU → Linear(256→43)`.

**Формулы слоёв**

* Свёртка: $y_{k}(i,j) = b_k + \sum_{c=1}^{C_{in}} \sum_{u=0}^{2} \sum_{v=0}^{2} w_{k,c,u,v}\, x_c(i+u-1,\, j+v-1)$
* Размер выхода свёртки: $H_{out} = \frac{H_{in} + 2p - k}{s} + 1 = \frac{32 + 2 - 3}{1} + 1 = 32$
* ReLU: $f(z) = \max(0, z)$; tanh: $f(z) = \frac{e^z - e^{-z}}{e^z + e^{-z}}$; сигмоида: $f(z) = \frac{1}{1+e^{-z}}$
* MaxPool 2×2: $y(i,j) = \max_{u,v \in \{0,1\}} x(2i+u,\, 2j+v)$
* Полносвязный слой: $y = Wx + b$
* Softmax: $p_k = \frac{e^{z_k}}{\sum_{j=1}^{43} e^{z_j}}$; кросс-энтропия: $L = -\ln p_{y}$, где $y$ — истинный класс
* Число параметров свёртки: $(C_{in} \cdot k^2 + 1)\cdot C_{out}$; полносвязного слоя: $(N_{in}+1)\cdot N_{out}$

In [ ]:
baseline_kwargs = dict(channels=(32, 64, 128), activation="relu", batch_norm=False, dropout=0.0)
baseline_model = TrafficSignCNN(**baseline_kwargs).to(device)
print(baseline_model)
summary_table = layer_summary(baseline_model)
summary_table.to_csv(config.RESULTS_DIR / "layer_summary_baseline.csv", index=False)
summary_table

In [ ]:
# Ручная проверка формул на реальном изображении: считаем вручную одно значение
# первой свёртки, softmax и кросс-энтропию и сравниваем с результатом PyTorch.
set_seed(config.SEED)
check_model = TrafficSignCNN(**baseline_kwargs).eval()
image, label = val_dataset[0]
conv = check_model.features[0]
i, j, k = 10, 12, 0   # позиция (строка, столбец) и номер фильтра
padded = torch.nn.functional.pad(image, (1, 1, 1, 1))
patch = padded[:, i:i + 3, j:j + 3]                     # окно 3x3 по всем 3 каналам
manual_value = (conv.weight[k] * patch).sum() + conv.bias[k]
with torch.no_grad():
    torch_value = conv(image.unsqueeze(0))[0, k, i, j]
print(f"Свёртка: вручную {manual_value.item():.6f}, PyTorch {torch_value.item():.6f}")
print(f"После ReLU: max(0, {manual_value.item():.4f}) = {max(0.0, manual_value.item()):.4f}")

with torch.no_grad():
    logits = check_model(image.unsqueeze(0))[0]
manual_probs = torch.exp(logits) / torch.exp(logits).sum()
manual_loss = -torch.log(manual_probs[label])
torch_loss = torch.nn.functional.cross_entropy(logits.unsqueeze(0), label.unsqueeze(0))
print(f"Кросс-энтропия необученной сети: вручную {manual_loss.item():.4f}, PyTorch {torch_loss.item():.4f}, "
      f"теоретически для случайного угадывания ln(43) = {np.log(43):.4f}")

## 5. Базовая модель

ReLU, без BatchNorm и Dropout, без аугментации, Adam с шагом обучения 0.001, размер пакета 128.
Метрика при подборе — доля верных ответов (accuracy) на валидационной выборке; на тестовой выборке
дополнительно считаются точность (precision), полнота (recall) и F1-мера по классам, т.к. классы несбалансированы.

In [ ]:
experiments = {}
model, experiments["01_baseline_relu"] = run_experiment(
    "01_baseline_relu", train_loader_plain, val_loader, baseline_kwargs, epochs=EXPERIMENT_EPOCHS, verbose=True)
plots.plot_training_history(experiments["01_baseline_relu"]["history"],
                            "Базовая модель (ReLU, без регуляризации)", "07_baseline_history.png")

## 6. Эксперименты

Каждый эксперимент меняет **одну** настройку относительно предыдущего шага, чтобы было видно её влияние.
Seed фиксирован, поэтому начальные веса и порядок пакетов одинаковы.

### 6.1. Функция активации

In [ ]:
for activation in ["tanh", "sigmoid"]:
    name = f"02_activation_{activation}"
    _, experiments[name] = run_experiment(name, train_loader_plain, val_loader,
                                          {**baseline_kwargs, "activation": activation}, epochs=EXPERIMENT_EPOCHS)
plots.plot_experiment_comparison(
    {"ReLU": experiments["01_baseline_relu"], "tanh": experiments["02_activation_tanh"],
     "Сигмоида": experiments["02_activation_sigmoid"]},
    "Сравнение функций активации", "08_activation_comparison.png")

### 6.2. Регуляризация: BatchNorm, Dropout, аугментация

In [ ]:
bn_kwargs = {**baseline_kwargs, "batch_norm": True}
_, experiments["03_batchnorm"] = run_experiment("03_batchnorm", train_loader_plain, val_loader, bn_kwargs,
                                                epochs=EXPERIMENT_EPOCHS)
bn_dropout_kwargs = {**bn_kwargs, "dropout": 0.5}
_, experiments["04_batchnorm_dropout"] = run_experiment("04_batchnorm_dropout", train_loader_plain, val_loader,
                                                        bn_dropout_kwargs, epochs=EXPERIMENT_EPOCHS)
_, experiments["05_bn_dropout_augmentation"] = run_experiment("05_bn_dropout_augmentation", train_loader_augmented,
                                                              val_loader, bn_dropout_kwargs, epochs=EXPERIMENT_EPOCHS)
plots.plot_experiment_comparison(
    {"Базовая": experiments["01_baseline_relu"], "+ BatchNorm": experiments["03_batchnorm"],
     "+ BatchNorm + Dropout": experiments["04_batchnorm_dropout"],
     "+ BatchNorm + Dropout + аугментация": experiments["05_bn_dropout_augmentation"]},
    "Влияние регуляризации на точность", "09_regularization_comparison.png")
plots.plot_experiment_comparison(
    {"Базовая": experiments["01_baseline_relu"],
     "+ BatchNorm + Dropout + аугментация": experiments["05_bn_dropout_augmentation"]},
    "Потери на валидации: признаки переобучения", "09b_regularization_val_loss.png", metric="val_loss")

### 6.3. Скорость обучения (learning rate)

In [ ]:
for lr in [1e-2, 1e-4]:
    name = f"06_lr_{lr:g}"
    _, experiments[name] = run_experiment(name, train_loader_augmented, val_loader, bn_dropout_kwargs,
                                          epochs=EXPERIMENT_EPOCHS, lr=lr)
plots.plot_experiment_comparison(
    {"lr = 0.01": experiments["06_lr_0.01"], "lr = 0.001": experiments["05_bn_dropout_augmentation"],
     "lr = 0.0001": experiments["06_lr_0.0001"]},
    "Влияние скорости обучения", "09c_learning_rate_comparison.png")

### 6.4. Глубина сети и время обучения

Сравниваются сети с 2, 3 и 4 свёрточными блоками. Для каждой считаются число параметров,
число операций умножения-сложения (MAC) на одно изображение и измеренное время эпохи.
Теоретическая оценка вычислений на эпоху: прямой проход ≈ 2·MAC операций с плавающей точкой,
обратный ≈ вдвое больше, итого ≈ 6·MAC·N на эпоху, где N — число обучающих изображений.

In [ ]:
depth_configs = {2: (32, 64), 3: (32, 64, 128), 4: (32, 64, 128, 256)}
for depth in [2, 4]:
    name = f"07_depth_{depth}_blocks"
    _, experiments[name] = run_experiment(name, train_loader_augmented, val_loader,
                                          {**bn_dropout_kwargs, "channels": depth_configs[depth]},
                                          epochs=EXPERIMENT_EPOCHS)
experiments["07_depth_3_blocks"] = experiments["05_bn_dropout_augmentation"]

rows = []
for depth, channels in depth_configs.items():
    table = layer_summary(TrafficSignCNN(**{**bn_dropout_kwargs, "channels": channels}).to(device))
    macs = table.iloc[-1]["Операции MAC"]
    summary = experiments[f"07_depth_{depth}_blocks"]
    rows.append({"Свёрточных блоков": depth, "Параметры": int(table.iloc[-1]["Параметры"]),
                 "Операции MAC, млн": macs / 1e6,
                 "Теор. GFLOP на эпоху": 6 * macs * len(train_dataset_augmented) / 1e9,
                 "Время эпохи, c": summary["mean_epoch_time_sec"],
                 "Точность на валидации": summary["best_val_accuracy"]})
depth_table = pd.DataFrame(rows)
depth_table.to_csv(config.RESULTS_DIR / "depth_vs_time.csv", index=False)
plots.plot_depth_cost(depth_table)
depth_table.round(4)

### 6.5. Сводная таблица экспериментов

In [ ]:
experiments_table = pd.DataFrame([{
    "Эксперимент": name, "Активация": s["model_kwargs"]["activation"],
    "BatchNorm": s["model_kwargs"].get("batch_norm", False), "Dropout": s["model_kwargs"].get("dropout", 0.0),
    "Аугментация": s["augmentation"], "Блоков": len(s["model_kwargs"]["channels"]), "lr": s["lr"],
    "Параметры": s["parameters"], "Лучшая эпоха": s["best_epoch"],
    "Точность train": round(s["final_train_accuracy"], 4),
    "Точность validation": round(s["best_val_accuracy"], 4),
    "Время эпохи, c": round(s["mean_epoch_time_sec"], 1)} for name, s in sorted(experiments.items())])
experiments_table.to_csv(config.RESULTS_DIR / "experiments_summary.csv", index=False)
experiments_table

## 7. Итоговая модель и оценка на тестовой выборке

Берём лучшую конфигурацию по валидации и обучаем дольше, с постепенным уменьшением шага обучения
(косинусное расписание). Тестовая выборка используется **один раз** — для итоговой оценки.

In [ ]:
candidates = {k: v for k, v in experiments.items() if not k.startswith("07_depth_3")}
best_name = max(candidates, key=lambda k: candidates[k]["best_val_accuracy"])
best = candidates[best_name]
final_kwargs = best["model_kwargs"]
final_loader = train_loader_augmented if best["augmentation"] else train_loader_plain
print("Лучшая конфигурация:", best_name, final_kwargs, "lr =", best["lr"])

final_model, final_summary = run_experiment("10_final_model", final_loader, val_loader, final_kwargs,
                                            epochs=FINAL_EPOCHS, lr=best["lr"], scheduler="cosine", verbose=True)
plots.plot_training_history(final_summary["history"], "Обучение итоговой модели", "11_final_history.png")
torch.save({"state_dict": final_model.state_dict(), "model_kwargs": final_kwargs, "mean": mean, "std": std},
           config.RESULTS_DIR / "final_model.pt")

In [ ]:
test_result = evaluate(final_model, test_loader, torch.nn.CrossEntropyLoss(), device, return_outputs=True)
y_true, probs = test_result["labels"], test_result["probs"]
y_pred = probs.argmax(axis=1)

macro_f1 = f1_score(y_true, y_pred, average="macro")
weighted_f1 = f1_score(y_true, y_pred, average="weighted")
print(f"Тестовая выборка: {len(y_true)} изображений")
print(f"Accuracy (доля верных ответов): {test_result['accuracy']:.4f}  -> ошибок: {(y_true != y_pred).sum()}")
print(f"Macro-F1 (среднее F1 по классам, все классы равноправны): {macro_f1:.4f}")
print(f"Weighted-F1 (F1 с весами по числу примеров класса):     {weighted_f1:.4f}")

precision, recall, f1, support = precision_recall_fscore_support(y_true, y_pred, labels=range(config.NUM_CLASSES),
                                                                 zero_division=0)
per_class = pd.DataFrame({"Класс": config.CLASS_NAMES_RU, "Precision": precision, "Recall": recall,
                          "F1": f1, "Изображений": support}).round(4)
per_class.to_csv(config.RESULTS_DIR / "test_per_class_metrics.csv", index_label="Номер")
with open(config.RESULTS_DIR / "test_metrics.json", "w") as f:
    json.dump({"accuracy": test_result["accuracy"], "macro_f1": macro_f1, "weighted_f1": weighted_f1,
               "errors": int((y_true != y_pred).sum()), "best_config": best_name}, f, ensure_ascii=False, indent=2)
per_class.sort_values("F1").head(10)

In [ ]:
matrix = plots.plot_confusion(y_true, y_pred)
plots.plot_per_class_f1(f1)
worst_classes = list(np.argsort(f1)[:4])
plots.plot_roc(y_true, probs, worst_classes)
plots.plot_misclassified(test_arrays.images, y_true, y_pred, probs)

# Самые частые пары «истинный класс -> предсказанный»
off_diagonal = [(matrix[t, p], t, p) for t in range(config.NUM_CLASSES) for p in range(config.NUM_CLASSES) if t != p]
top_pairs = sorted(off_diagonal, reverse=True)[:10]
pd.DataFrame([{"Истинный класс": f"{t}: {config.CLASS_NAMES_RU[t]}", "Предсказан как": f"{p}: {config.CLASS_NAMES_RU[p]}",
               "Доля изображений класса": round(v, 4)} for v, t, p in top_pairs])

## Итоги

Все графики сохранены в `results/figures/`, таблицы — в `results/`, история каждого эксперимента — в
`results/experiments/`. Эти файлы используются в отчёте.

In [ ]:
# Архив с результатами (графики, таблицы, модель) — скачать и прислать для написания отчёта
import shutil
archive = shutil.make_archive("gtsrb_results", "zip", root_dir=".", base_dir="results")
print("Архив:", archive)
if "google.colab" in sys.modules:
    from google.colab import files
    files.download(archive)